# MedBridge — Hindi and Spanish medical LoRA demo (NLLB-200)

This edited copy uses `medical_training_pairs.csv` (60 synthetic bilingual pairs), both translation directions, an 80/20 group split, and 40 optimization steps. **These translations are illustrative and have not been clinically or professionally reviewed.** This is a training demonstration, not a medically validated interpreter. Keep the original model available and test medication numbers, negation, and allergy statements before a demo. Do not use identifiable patient data.


## 0. Install (Terminal, once)
Create a separate environment so the live app remains unaffected. Run the following in Terminal, then choose the new `medbridge-lora` kernel in Jupyter.

In [1]:
# Terminal commands (do not run this cell as Python):
# python3 -m venv .venv-lora
# source .venv-lora/bin/activate
# python -m pip install --upgrade pip
# python -m pip install 'transformers>=4.40,<5' 'peft>=0.10' torch sentencepiece pandas ipykernel notebook
# python -m ipykernel install --user --name medbridge-lora --display-name 'medbridge-lora'
# jupyter notebook

## 1. Load the supplied synthetic Hindi/Spanish medical pairs and split before reversing directions


In [2]:
from pathlib import Path
import pandas as pd
import random, torch
random.seed(42); torch.manual_seed(42)
LANG = {'en':'eng_Latn','es':'spa_Latn','hi':'hin_Deva'}
DATA_PATH = Path('medical_training_pairs.csv')  # synthetic demonstration examples; review before clinical use
SMOKE_TEST = False  # demo fine-tuning only; NOT a clinical accuracy claim
raw = pd.read_csv(DATA_PATH).dropna()
required={'src_lang','tgt_lang','source','target'}
assert required.issubset(raw.columns)
raw = raw.drop_duplicates(subset=['src_lang','tgt_lang','source','target']).reset_index(drop=True)
assert all(x in LANG for x in raw.src_lang) and all(x in LANG for x in raw.tgt_lang)
assert all(raw.src_lang != raw.tgt_lang)
assert all(raw.source.str.strip()) and all(raw.target.str.strip())
print('Original parallel pairs:',len(raw))
print(raw.groupby(['src_lang','tgt_lang']).size())
if not SMOKE_TEST:
    assert len(raw)>=40, 'Expected at least 40 parallel examples'
    assert all(raw.groupby('tgt_lang').size().reindex(['es','hi'],fill_value=0)>=20)

# Split original pairs FIRST, before reversing, to prevent reverse-direction leakage.
indices=list(range(len(raw))); random.shuffle(indices)
cut=max(1,int(len(indices)*0.8)); cut=min(cut,len(indices)-1)
train_original=raw.iloc[indices[:cut]]
val_original=raw.iloc[indices[cut:]]
def bidirectional(frame):
    forward=frame.copy()
    reverse=frame.rename(columns={'src_lang':'tgt_lang','tgt_lang':'src_lang','source':'target','target':'source'})
    return pd.concat([forward,reverse],ignore_index=True)
train=bidirectional(train_original).to_dict('records')
val=bidirectional(val_original).to_dict('records')
print('Train directional examples:',len(train),'Validation:',len(val))
print('Example:',train[0])

Original parallel pairs: 60
src_lang  tgt_lang
en        es          30
          hi          30
dtype: int64
Train directional examples: 96 Validation: 24
Example: {'src_lang': 'en', 'tgt_lang': 'hi', 'source': 'Do you have diabetes?', 'target': 'क्या आपको मधुमेह है?'}


## 2. Load NLLB and attach LoRA

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import LoraConfig, TaskType, get_peft_model
MODEL_NAME='facebook/nllb-200-distilled-600M'
DEVICE='mps' if torch.backends.mps.is_available() else 'cpu'
print('Device:',DEVICE)
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
base=AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
base.config.use_cache=False
config=LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM,r=4,lora_alpha=8,lora_dropout=0.05,bias='none',target_modules=['q_proj','v_proj'])
model=get_peft_model(base,config).to(DEVICE)
model.print_trainable_parameters()

/Users/siapatelpuri/Desktop/Sachacks/MedBridge/.venv-lora/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps
trainable params: 589,824 || all params: 615,663,616 || trainable%: 0.0958


## 3. Encode source and target using the correct NLLB language tags

In [4]:
MAX_LENGTH=80
def encode_example(row):
    tokenizer.src_lang=LANG[row['src_lang']]
    tokenizer.tgt_lang=LANG[row['tgt_lang']]
    item=tokenizer(row['source'],max_length=MAX_LENGTH,truncation=True,return_tensors='pt')
    target=tokenizer(text_target=row['target'],max_length=MAX_LENGTH,truncation=True,return_tensors='pt')
    return {'input_ids':item['input_ids'].to(DEVICE),
            'attention_mask':item['attention_mask'].to(DEVICE),
            'labels':target['input_ids'].to(DEVICE)}
encoded_train=[encode_example(r) for r in train]
encoded_val=[encode_example(r) for r in val]
print('Encoded:',len(encoded_train),'train examples')

Encoded: 96 train examples


## 4. Run a small LoRA training loop (batch size 1)

In [5]:
import time
from torch.optim import AdamW
MAX_STEPS=2 if SMOKE_TEST else 40  # tune after confirming runtime and data quality
LR=2e-4
optimizer=AdamW((p for p in model.parameters() if p.requires_grad),lr=LR)
model.train(); start=time.time()
for step in range(MAX_STEPS):
    sample=encoded_train[step%len(encoded_train)]
    optimizer.zero_grad(set_to_none=True)
    loss=model(**sample).loss
    loss.backward()
    torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad),1.0)
    optimizer.step()
    print(f'Step {step+1}/{MAX_STEPS}: loss={loss.item():.4f} elapsed={time.time()-start:.1f}s')
print('Training finished. This is a smoke test, not a medical-quality model.' if SMOKE_TEST else 'Training finished; evaluate against baseline on held-out data.')

Step 1/40: loss=0.9682 elapsed=51.1s
Step 2/40: loss=0.5995 elapsed=71.9s
Step 3/40: loss=1.3393 elapsed=82.6s
Step 4/40: loss=1.1634 elapsed=107.8s
Step 5/40: loss=1.0627 elapsed=113.3s
Step 6/40: loss=0.8132 elapsed=114.6s
Step 7/40: loss=1.3883 elapsed=114.9s
Step 8/40: loss=1.3520 elapsed=115.4s
Step 9/40: loss=1.1904 elapsed=127.4s
Step 10/40: loss=1.7594 elapsed=129.1s
Step 11/40: loss=1.3047 elapsed=129.2s
Step 12/40: loss=1.1957 elapsed=129.4s
Step 13/40: loss=1.6678 elapsed=129.6s
Step 14/40: loss=1.0081 elapsed=129.8s
Step 15/40: loss=1.9504 elapsed=129.9s
Step 16/40: loss=2.1678 elapsed=130.1s
Step 17/40: loss=0.6258 elapsed=130.2s
Step 18/40: loss=0.7280 elapsed=131.0s
Step 19/40: loss=1.4376 elapsed=131.2s
Step 20/40: loss=0.6781 elapsed=131.3s
Step 21/40: loss=0.8027 elapsed=131.6s
Step 22/40: loss=0.8826 elapsed=131.8s
Step 23/40: loss=1.6468 elapsed=131.9s
Step 24/40: loss=0.9052 elapsed=132.1s
Step 25/40: loss=1.9191 elapsed=132.2s
Step 26/40: loss=0.5414 elapsed=132.4

## 5. Save adapter, test translations, and check validation loss

In [6]:
ADAPTER_DIR=Path('adapters/medical_multilingual_smoke' if SMOKE_TEST else 'adapters/medical_multilingual')
ADAPTER_DIR.mkdir(parents=True,exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Saved adapter:',ADAPTER_DIR.resolve())
model.eval()
@torch.inference_mode()
def translate(text,src,tgt):
    tokenizer.src_lang=LANG[src]
    x=tokenizer(text,return_tensors='pt',truncation=True,max_length=MAX_LENGTH).to(DEVICE)
    ids=model.generate(**x,forced_bos_token_id=tokenizer.convert_tokens_to_ids(LANG[tgt]),max_new_tokens=80,num_beams=2)
    return tokenizer.batch_decode(ids,skip_special_tokens=True)[0]
for lang in ['es','hi']:
    print(lang,translate('Do not take two tablets at once.','en',lang))
losses=[]
with torch.inference_mode():
    for item in encoded_val:
        losses.append(model(**item).loss.item())
print('Held-out loss (small set; not a clinical metric):',sum(losses)/len(losses))

Saved adapter: /Users/siapatelpuri/Desktop/Sachacks/MedBridge/MedBridge_LoRA_edited/adapters/medical_multilingual


/Users/siapatelpuri/Desktop/Sachacks/MedBridge/.venv-lora/lib/python3.12/site-packages/transformers/generation/utils.py:1733: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


es No tome dos comprimidos a la vez.
hi एक ही समय में दो गोलियां न लें।
Held-out loss (small set; not a clinical metric): 0.732651182760795


## 6. Verify saved adapter

The adapter is saved to `adapters/medical_multilingual/`. This notebook does not automatically change the live app. The live translator must explicitly load the adapter using PEFT.


In [7]:
from pathlib import Path
import json

# ADAPTER_DIR is set in Section 5. This also works after a kernel restart.
adapter_dir = Path(globals().get("ADAPTER_DIR", "adapters/medical_multilingual_smoke"))
config_file = adapter_dir / "adapter_config.json"
weights_file = next((p for p in [adapter_dir / "adapter_model.safetensors", adapter_dir / "adapter_model.bin"] if p.exists()), None)

assert config_file.is_file(), f"Missing adapter config: {config_file}"
assert weights_file is not None, f"Missing adapter weights in {adapter_dir}"

config = json.loads(config_file.read_text())
print("Adapter verified:", adapter_dir.resolve())
print("Base model:", config.get("base_model_name_or_path"))
print("Weights:", weights_file.name)
print("To test in server.py, load the adapter with PeftModel.from_pretrained(base, str(adapter_dir)).")
print("This smoke-test adapter has NOT been clinically validated.")


Adapter verified: /Users/siapatelpuri/Desktop/Sachacks/MedBridge/MedBridge_LoRA_edited/adapters/medical_multilingual
Base model: facebook/nllb-200-distilled-600M
Weights: adapter_model.safetensors
To test in server.py, load the adapter with PeftModel.from_pretrained(base, str(adapter_dir)).
This smoke-test adapter has NOT been clinically validated.
